In [ ]:
!pip install pillow
!pip install geopandas
!pip install rasterio --upgrade
!pip install scipy -U
!pip install mkl
!pip install fiona
!pip install C:\Users\flopes1\Downloads\GDAL-3.4.3-cp39-cp39-win_amd64.whl
!pip install ogr

# Convert raster image to a set of tiles

In [ ]:
import geopandas as gpd
import rasterio
import rasterio.features
import numpy as np
from tqdm import tqdm
from osgeo import gdal, ogr

# Abrir a imagem TIFF e o arquivo shapefile
tiff_file = gdal.Open(r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\08-30-2021\LARGO1_0830.tif")
shp_file = ogr.Open(r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\PLOTBOUNDARIES\PLOTBOUNDARIES_PROJECT.shp")
layer = shp_file.GetLayer()

# Criar um novo arquivo TIFF para cada feição do shapefile
for i in tqdm(range(layer.GetFeatureCount())):
    feature = layer.GetFeature(i)
    geometry = feature.GetGeometryRef()
    xmin, xmax, ymin, ymax = geometry.GetEnvelope()

    # Definir as opções do recorte
    options = gdal.WarpOptions(
        outputBounds=(xmin, ymin, xmax, ymax),
        format='GTiff',
        cutlineDSName=r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\PLOTBOUNDARIES\PLOTBOUNDARIES_PROJECT.shp",
        cropToCutline=True,
        cutlineWhere=f"id='{feature.GetField('id')}'"
    )

    # Salvar a imagem recortada em um novo arquivo TIFF
    gdal.Warp(f"imagem_segmentada_{feature.GetField('id')}_{feature.GetField('HEALTH_STA')}.tif", tiff_file, options=options)


# Use the tiles as input for training a GAN

### Define o conjunto de dados a ser utilizado

In [ ]:
!pip install tiff
!pip install torch
!pip install torchvision
!pip install pytorch-lightning


In [ ]:
!pip install argparse

In [ ]:
!pip install libtiff

In [ ]:
!pip install tifffile

In [ ]:
!pip install opencv-python

In [ ]:
!pip install opencv-torchvision-transforms-yuzhiyang --user

In [2]:
!pip install tensorflow

  Using cached tensorflow-2.11.0-cp39-cp39-win_amd64.whl (1.9 kB)
  Using cached tensorflow_intel-2.11.0-cp39-cp39-win_amd64.whl (266.3 MB)
  Using cached google_pasta-0.2.0-py3-none-any.whl (57 kB)
  Using cached tensorboard-2.11.2-py3-none-any.whl (6.0 MB)
  Using cached flatbuffers-23.3.3-py2.py3-none-any.whl (26 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl (12 kB)
  Using cached gast-0.4.0-py3-none-any.whl (9.8 kB)
  Using cached tensorflow_io_gcs_filesystem-0.31.0-cp39-cp39-win_amd64.whl (1.5 MB)
  Using cached absl_py-1.4.0-py3-none-any.whl (126 kB)
  Using cached libclang-15.0.6.1-py2.py3-none-win_amd64.whl (23.2 MB)
  Using cached termcolor-2.2.0-py3-none-any.whl (6.6 kB)
  Using cached opt_einsum-3.3.0-py3-none-any.whl (65 kB)
  Using cached tensorflow_estimator-2.11.0-py2.py3-none-any.whl (439 kB)
  Using cached keras-2.11.0-py2.py3-none-any.whl (1.7 MB)
  Using cached grpcio-1.51.3-cp39-cp39-win_amd64.whl (3.7 MB)
  Using cached Markdown-3.4.1-py3-none-any.whl (9

In [10]:
from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path
import tifffile as tiff
import cv2 as cv 
import numpy as np
import base64

def data_uri_to_cv2_img(uri):
    print(uri)
    encoded_data = uri
    nparr = np.frombuffer(uri, np.uint16)
    img = cv.imdecode(nparr, cv.IMREAD_COLOR)
    print(img)
    return img

class MyDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # Recupera as imagens e os rótulos
        self.images = []  # Lista de caminhos para as imagens
        self.labels = []  # Lista de rótulos correspondentes

        # Percorre o diretório raiz e adiciona as imagens e os rótulos às listas
        # Aqui é assumido que as imagens estão organizadas em pastas separadas para cada classe
        # com o nome da pasta correspondendo ao rótulo
        for label in os.listdir(self.root_dir):
            if label.endswith('.tif'):
                image_path = os.path.join(self.root_dir, label)
                self.images.append(image_path)
                # Extrai o rótulo do nome do arquivo usando o método split() da classe str
                label_str = label.split('_')[-1]  # Supondo que o rótulo esteja no final do nome do arquivo
                
                self.labels.append(os.path.splitext(label_str)[0])
                        

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        label = self.labels[idx]
        image = tiff.imread(image_path)
        #image = cv.imread(image_path, -1)
        #print(f'dtype: {image.dtype}, shape: {image.shape}, min: {np.min(image)}, max: {np.max(image)}')
        # Abre a imagem usando o PIL e aplica transformações, se especificado
        #image = Image.open(image_path, formats=['TIFF'])
        #image = np.array(tifffile.imread(image_path))
        #image = Image.fromarray(image)
        
        #p = Path(image_path)
        #p.as_uri()
        

        #data = open(image_path, 'rb').read()
        #image = data_uri_to_cv2_img(data)
        #cv.imread(img)
        #print(os.path.exists(image_path))
        #print(image_path)
        
        #image = cv.imread(f'{str(image_path)}', cv.IMREAD_UNCHANGED)
        image = (image - 2**15).astype(np.int16)
        if self.transform:
            image = self.transform(image)

        return image, label


In [12]:
import os
from os import makedirs

# PyTorch packages
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils

from PIL import Image

# Typing
from torch.utils.data.dataloader import DataLoader
from torch.optim.optimizer import Optimizer
from torch.utils.data import Dataset
from torch.functional import Tensor
from typing import Dict, Tuple, List

from cvtorchvision import cvtransforms

# PyTorch Lightning
import pytorch_lightning as pl

# Output Folder for the files
path = './output_lightning'

# create output folder if doesn't exist
makedirs(path, exist_ok=True)

# shape of the image (gray)
img_shape = (1, 462, 312)


# Generator Model
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super(Generator, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 64*29*20),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(32, 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(16, 5, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), 64, 29, 20)
        x = self.deconv(x)
        return x

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.conv1 = nn.Conv2d(5, 16, 3, stride=2, padding=1)  # 233x156
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1)  # 117x78
        self.conv3 = nn.Conv2d(32, 64, 3, stride=2, padding=1)  # 59x40
        self.conv4 = nn.Conv2d(64, 1, 3, stride=1, padding=1)  # 59x40
        
    def forward(self, x):
        x = F.leaky_relu(self.conv1(x), 0.2)
        x = F.leaky_relu(self.conv2(x), 0.2)
        x = F.leaky_relu(self.conv3(x), 0.2)
        x = torch.sigmoid(self.conv4(x))
        x = x.view(x.size(0), -1)
        return x


# Lightning Module
class GAN(pl.LightningModule):
    def __init__(self, hparams) -> None:
        super(GAN, self).__init__()

        self.hparams.lr = hparams.lr
        self.hparams.batch_size = hparams.batch_size
        self.generator = Generator()
        self.discriminator = Discriminator()

    def forward(self, x) -> Tensor:
        return self.discriminator(x)

    def loss_function(self, y_hat, y) -> Tensor:
        return nn.BCELoss()(y_hat, y)

    def configure_optimizers(self) -> Tuple[List[Optimizer], List]:
        optimizer_G = torch.optim.Adam(self.generator.parameters(), lr=self.hparams.lr, betas=(0.4, 0.999))
        optimizer_D = torch.optim.Adam(self.discriminator.parameters(), lr=self.hparams.lr, betas=(0.4, 0.999))

        return [optimizer_G, optimizer_D], []

    def prepare_data(self) -> Dataset:
        transform = cvtransforms.Compose([cvtransforms.ToTensor(),
                                        cvtransforms.Normalize([0.5], [0.5])])
        transform = cvtransforms.Compose([
            cvtransforms.Resize((466, 312)),
            cvtransforms.ToTensor()
        ])

        # Instancia o dataset com o diretório raiz "C:/my_dataset"
        train_data = MyDataset(r'C:/Users/flopes1/OneDrive - Saint Louis University/Desktop/Repos/plant-disease-prediction/',
                               transform=transform)
        print(train_data)
        """
            train_data = datasets.MNIST('./data',
                                    train=True,
                                    download=False,
                                    transform=transform)
        """
        return train_data

    def train_dataloader(self) -> DataLoader:
        train_data = self.prepare_data()
        train_loader = DataLoader(train_data,
                                  batch_size=self.hparams.batch_size,
                                  shuffle=True)
        return train_loader

    def training_step(self, batch, batch_idx, optimizer_idx) -> Dict:
        real_images, _ = batch
        valid = torch.ones(real_images.size(0), 1)
        fake = torch.zeros(real_images.size(0), 1)
        criterion = self.loss_function

        if optimizer_idx == 0:
            gen_input = torch.randn(real_images.shape[0], 100)
            self.gen_images = self.generator(gen_input)

            g_loss = criterion(
                self(self.gen_images), valid)

            tqdm_dict = {'g_loss': g_loss}
            output = {
                'loss': g_loss,
                'progress_bar': tqdm_dict,
                'log': tqdm_dict,
                'g_loss': g_loss
            }
            return output

        if optimizer_idx == 1:
            real_loss = criterion(
                self(real_images), valid)
            fake_loss = criterion(
                self(self.gen_images.detach()), fake)
            d_loss = (real_loss + fake_loss) / 2.0

            tqdm_dict = {'d_loss': d_loss}
            output = {
                'loss': d_loss,
                'progress_bar': tqdm_dict,
                'log': tqdm_dict,
                'd_loss': d_loss
            }
            return output

    def on_train_epoch_end(self) -> None:
        utils.save_image(self.gen_images.data[:25],
                         path + '/%d.tif' % self.current_epoch,
                         nrow=5,
                         padding=0,
                         normalize=True)


from argparse import ArgumentParser

# Hyperparameters
parser = ArgumentParser(description='GAN Wheat dataset')
parser = pl.Trainer.add_argparse_args(parser)
parser.add_argument('--batch_size', type=int, default=32)
parser.add_argument('--lr', type=float, default=2e-4)
parser.add_argument('-f', required=False)

args = parser.parse_args()

# Model Initialization
gan = GAN(hparams=args)

delattr(args, 'f')
# Model Training
trainer = pl.Trainer.from_argparse_args(args,
                                        max_epochs=20,
                                        fast_dev_run=True)

trainer.fit(gan)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.

  | Name          | Type          | Params
------------------------------------------------
0 | generator     | Generator     | 3.8 M 
1 | discriminator | Discriminator | 24.4 K
------------------------------------------------
3.8 M     Trainable params
0         Non-trainable params
3.8 M     Total params
15.264    Total estimated model params size (MB)


Training: 0it [00:00, ?it/s]

ValueError: Using a target size (torch.Size([32, 1])) that is different to the input size (torch.Size([32, 580])) is deprecated. Please ensure they have the same size.

In [ ]:
from torchvision import transforms

# Cria uma transformação que dimensiona as imagens para 466x312 pixels e as converte em tensores
transform = transforms.Compose([
    transforms.Resize((466, 312)),
    transforms.ToTensor()
])

# Instancia o dataset com o diretório raiz "C:/my_dataset"
dataset = MyDataset(root_dir='C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction', transform=transform)

# Usa o DataLoader para carregar os dados em lotes de tamanho 32, embaralhando os dados para evitar overfitting
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)